In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math, os
import pandas as pd
import numpy as np
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import random

In [ ]:
def data_to_model(data, sequence_length=120, step_size=25):
    data = torch.tensor(data, dtype=torch.float32)
    sequences = []
    N = data.shape[0]
    
    for start_idx in range(0, N - sequence_length + 1, step_size):
        sequence = data[start_idx:start_idx + sequence_length]
        sequences.append(sequence)
    
    sequences = torch.stack(sequences)
    
    return sequences

def get_data(mode = '1-hover', phase = '2_5'):
    # 1-hover 2-hover-accx-flag3-params03 4-hover-motor4-flag4-params085 
    fmode = mode
    base_path = "XXXXXXXX" # change to your path

    sycn_raw_data = f'all_sycn_raw_data_mode{phase}.csv'
    sycn_sim_data = f'all_sycn_sim_data_mode{phase}.csv'

    sycn_raw_data_path = os.path.join(base_path, 'data', 'Dpro', fmode, sycn_raw_data)
    sycn_sim_data_path = os.path.join(base_path, 'data', 'Dpro', fmode, sycn_sim_data)

    raw_data = pd.read_csv(sycn_raw_data_path).to_numpy()
    sim_data = pd.read_csv(sycn_sim_data_path).to_numpy()
    return raw_data, sim_data

def get_train_data(mode= ['1-hover', '2-forward', '1-hover'], phase = ['2_6', '2_5', '2_3']):
    fi_raw, fi_sim = get_data(mode= mode[0], phase=phase[0])
    sim_data = np.vstack((fi_sim))
    raw_data = np.vstack((fi_raw))
    for mo, ph in zip(mode[1:], phase[1:]):
        raw, sim = get_data(mode= mo, phase= ph)
        sim_data = np.vstack((sim_data, sim))
        raw_data = np.vstack((raw_data, raw))

    sim_data_split = np.split(sim_data, indices_or_sections=3, axis=1)
    raw_data_split = np.split(raw_data, indices_or_sections=3, axis=1)

    sim_gyro, sim_acc, sim_mag = sim_data_split[0], sim_data_split[1], sim_data_split[2]    
    raw_gyro, raw_acc, raw_mag = raw_data_split[0], raw_data_split[1], raw_data_split[2]

    sim_data_ = np.hstack((sim_acc, sim_gyro, sim_mag))
    raw_data_ = np.hstack((raw_acc, raw_gyro, raw_mag))

    return sim_data_, raw_data_

def get_sensor_data(mode= ['1-hover', '2-forward', '1-hover'], phase = ['2_6', '2_5', '2_3'], label = 1, shuffle=False, ratio=1, sequence_length=80, step_size=1, source='raw'):
    sim, raw = get_train_data(mode, phase)
    raw_acc, raw_gyro, raw_mag = raw[:,0:3], raw[:,3:6], raw[:,6:9]
    sim_acc, sim_gyro, sim_mag = sim[:,0:3], sim[:,3:6], sim[:,6:9]
    raw_ = np.hstack((raw_acc, raw_gyro))
    sim_ = np.hstack((sim_acc, sim_gyro))

    raw_2_model = data_to_model(data=raw_, sequence_length=sequence_length, step_size=step_size)
    raw_2_model_labels = torch.full((raw_2_model.size(0), 1), label, dtype=torch.long)

    sim_2_model = data_to_model(data=sim_, sequence_length=sequence_length, step_size=step_size)
    sim_2_model_labels = torch.full((sim_2_model.size(0), 1), label, dtype=torch.long)

    if shuffle:
        torch.manual_seed(42)
        indices = torch.randperm(raw_2_model.size(0))
        raw_2_model = raw_2_model[indices]
        raw_2_model_labels = raw_2_model_labels[indices]
        sim_2_model = sim_2_model[indices]
        sim_2_model_labels = sim_2_model_labels[indices]
    raw_2_model = raw_2_model[:int(raw_2_model.size(0) * ratio)]
    raw_2_model_labels = raw_2_model_labels[:int(raw_2_model_labels.size(0) * ratio)]
    sim_2_model = sim_2_model[:int(sim_2_model.size(0) * ratio)]
    sim_2_model_labels = sim_2_model_labels[:int(sim_2_model_labels.size(0) * ratio)]

    if source == 'raw':
        return raw_2_model, raw_2_model_labels
    else:
        return sim_2_model, sim_2_model_labels

In [ ]:
'''Step 1: Get [raw] or [sim] data'''
# 1-hover 2-hover-accx-flag3-params03 4-hover-motor4-flag4-params085 7-hover-motor3-flag3-params200
source = 'raw'
sequence_length, step_size, ratio = 80, 4, 1
normal, normal_lables = get_sensor_data(mode= ['1-hover'], phase = ['2_6','2_3'], label = 1, shuffle=True, ratio=ratio, sequence_length=sequence_length, step_size=step_size, source=source)
af3, af3_labels =  get_sensor_data(mode= ['2-hover-accx-flag3-params03'], phase = ['2_9'], label = 2, shuffle=True, ratio=ratio, sequence_length=sequence_length, step_size=step_size, source=source)
m1f4, m1f4_labels =  get_sensor_data(mode= ['7-hover-motor3-flag3-params200'], phase = ['2_9'], label = 3, shuffle=True, ratio=ratio, sequence_length=sequence_length, step_size=step_size, source=source)

print(f'normal:{normal.shape} af3:{af3.shape} m1f4:{m1f4.shape}')
print(f'normal_lables:{normal_lables.shape} af3_labels:{af3_labels.shape} m1f4_labels:{m1f4_labels.shape}')

data = torch.cat((normal, af3, m1f4), dim=0)
labels_ = torch.cat((normal_lables, af3_labels, m1f4_labels), dim=0).squeeze(1)

channel = data.shape[2]
num_classes = 3
one_hot = torch.zeros(labels_.size(0), num_classes, dtype=torch.float)
for i in range(labels_.size(0)):
    one_hot[i][labels_[i] - 1] = 1
labels = one_hot
print(f'data:{data.shape} labels:{labels.shape}')

In [ ]:
class ConvBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=None, stride=1, dropout=0.0):
        super().__init__()
        if padding is None:
            padding = (kernel_size - 1) // 2  # preserve length by default
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, stride=stride, padding=padding)
        self.bn = nn.BatchNorm1d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, x):
        # x: (B, C, T)
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.dropout(x)
        return x  # (B, out_ch, T')

class ConvBiLSTMEncoder(nn.Module):
    def __init__(self,
                 in_channels=6,
                 conv_filters=[48, 64, 128],
                 conv_kernels=[5, 5, 3],
                 conv_dropouts=[0.1, 0.1, 0.1],
                 lstm_hidden=128,
                 lstm_layers=2,
                 bidirectional=True,
                 dropout_lstm=0.2):
        super().__init__()

        assert len(conv_filters) == len(conv_kernels) == len(conv_dropouts)
        self.conv_blocks = nn.ModuleList()
        prev_ch = in_channels
        for out_ch, k, d in zip(conv_filters, conv_kernels, conv_dropouts):
            self.conv_blocks.append(ConvBlock1D(prev_ch, out_ch, kernel_size=k, padding=(k-1)//2, dropout=d))
            prev_ch = out_ch

        # Bi-LSTM expects input (T, B, feat). We'll feed conv features after permute.
        self.lstm_input_size = prev_ch  # because conv keeps time dimension as last dim
        self.lstm = nn.LSTM(input_size=self.lstm_input_size,
                            hidden_size=lstm_hidden,
                            num_layers=lstm_layers,
                            batch_first=True,  # we will use (B, T, feat)
                            bidirectional=bidirectional,
                            dropout=dropout_lstm if lstm_layers > 1 else 0.0)
        self.bidirectional = bidirectional
        self.lstm_hidden = lstm_hidden
        self.lstm_layers = lstm_layers

    def forward(self, x, return_sequence=True):
        # permute to (B, C, T) for conv1d
        x = x.permute(0, 2, 1)  # (B, C, T)
        for conv in self.conv_blocks:
            x = conv(x)  # maintain (B, C', T)

        # prepare for LSTM: (B, T, feat)
        x = x.permute(0, 2, 1)  # (B, T, feat)
        # Bi-LSTM
        lstm_out, (h_n, c_n) = self.lstm(x)  # lstm_out: (B, T, hidden * num_dir)
        # h_n: (num_layers * num_dirs, B, hidden)
        if self.bidirectional:
            # take last layer's forward and backward states and concat
            # index for last layer:
            idx_fwd = (self.lstm_layers - 1) * 2
            idx_bwd = idx_fwd + 1
            h_fwd = h_n[idx_fwd]  # (B, hidden)
            h_bwd = h_n[idx_bwd]  # (B, hidden)
            embedding = torch.cat([h_fwd, h_bwd], dim=-1)  # (B, hidden*2)
        else:
            embedding = h_n[-1]  # (B, hidden)

        if return_sequence:
            return embedding, lstm_out  # (B, emb_dim), (B, T, hidden*dirs)
        else:
            return embedding  # (B, emb_dim)

class FaultClassifier(nn.Module):
    def __init__(self,
                 encoder: ConvBiLSTMEncoder,
                 num_classes=6,
                 fc_hidden=[128],
                 dropout_fc=0.3):
        super().__init__()
        self.encoder = encoder

        # additional Bi-LSTM on top of encoder sequence output (paper: unrolled through Bi-LSTM)
        enc_out_dim = encoder.lstm_hidden * (2 if encoder.bidirectional else 1)
    
        # FC head
        fc_input_dim = enc_out_dim
        fc_layers = []
        prev = fc_input_dim
        for h in fc_hidden:
            fc_layers.append(nn.Linear(prev, h))
            fc_layers.append(nn.ReLU(inplace=True))
            fc_layers.append(nn.Dropout(dropout_fc))
            prev = h
        fc_layers.append(nn.Linear(prev, num_classes))
        self.fc = nn.Sequential(*fc_layers)

    def forward(self, x):
        """
        x: (B, T, C)
        returns logits (B, num_classes)
        """
        embedding, seq = self.encoder(x, return_sequence=True)
        logits = self.fc(embedding)  # (B, num_classes)
        return logits

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def train_one_epoch(model, device, dataloader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    n = 0
    for xb, yb in dataloader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return running_loss / n if n > 0 else 0.0

def evaluate_model(model, device, dataloader):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            logits = model(xb)
            prob = torch.softmax(logits, dim=-1)
            pred = torch.argmax(prob, dim=1).cpu().numpy()
            preds.append(pred)
            trues.append(torch.argmax(yb, dim=1).cpu().numpy()) 
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    return trues, preds

def training(model, train_loader, test_loader, optimizer, criterion, device, epochs):
    total_train_loss = []
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, device, train_loader, optimizer, criterion)
        trues_val, preds_val = evaluate_model(model, device, test_loader)
        metrics = compute_classification_metrics(trues_val, preds_val, labels=list(range(num_classes)))

        total_train_loss.append(train_loss)
        print(f"Epoch {epoch:02d} | TrainLoss {train_loss:.4f} | Val Precision {metrics['precision']:.4f} "
              f"| Val Recall {metrics['recall']:.4f} | Val F1 {metrics['f1']:.4f} | Val Acc {metrics['accuracy']:.4f}")

    trues_test, preds_test = evaluate_model(model, device, test_loader)
    final_metrics = compute_classification_metrics(trues_test, preds_test, labels=list(range(num_classes)))
    print("\nFinal test metrics (best model):")
    print("Confusion matrix:\n", final_metrics['confusion_matrix'])
    print(f"Precision (mean over classes): {final_metrics['precision']:.6f}")
    print(f"Recall    (mean over classes): {final_metrics['recall']:.6f}")
    print(f"F1 score  (derived)         : {final_metrics['f1']:.6f}")
    print(f"Accuracy (sum(TP)/sum(cm))  : {final_metrics['accuracy']:.6f}")

    return model, final_metrics, total_train_loss

def compute_classification_metrics(y_true, y_pred, labels=None):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    # cm: rows true, cols pred
    tp = np.diag(cm).astype(float)
    fp = np.sum(cm, axis=0).astype(float) - tp
    fn = np.sum(cm, axis=1).astype(float) - tp

    # avoid divide by zero: if denom==0 set per-class metric to 0
    with np.errstate(divide='ignore', invalid='ignore'):
        per_class_precision = np.divide(tp, (tp + fp))
        per_class_precision[np.isnan(per_class_precision)] = 0.0
        per_class_recall = np.divide(tp, (tp + fn))
        per_class_recall[np.isnan(per_class_recall)] = 0.0

    precision = np.mean(per_class_precision)
    recall = np.mean(per_class_recall)
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)
    accuracy = np.sum(tp) / np.sum(cm) if np.sum(cm) > 0 else 0.0

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'per_class_precision': per_class_precision,
        'per_class_recall': per_class_recall,
        'confusion_matrix': cm
    }

In [ ]:
random_state = 42
set_seed(random_state)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

test_size = 0.25
batch_size = 32
# convert one-hot labels to class indices for stratify
label_idx = torch.argmax(labels, dim=1).numpy()
train_x, test_x, train_y, test_y = train_test_split(
    data.numpy(), labels.numpy(), test_size=test_size, random_state=random_state, stratify=label_idx
)
print(f'train_x: {train_x.shape} test_x: {test_x.shape}')
print(f'train_y: {train_y.shape} test_y: {test_y.shape}')

train_dataset = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
test_dataset = TensorDataset(torch.from_numpy(test_x), torch.from_numpy(test_y))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

In [ ]:

encoder = ConvBiLSTMEncoder(in_channels=channel,
                            conv_filters=[48, 64, 96],
                            conv_kernels=[5,5,3],
                            conv_dropouts=[0.5, 0.5, 0.5],
                            lstm_hidden=256,
                            lstm_layers=1,
                            bidirectional=True,
                            dropout_lstm=0.3)

model = FaultClassifier(encoder=encoder,
                        num_classes=num_classes,
                        fc_hidden=[128],
                        dropout_fc=0.3)

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
model, metric, tloss = training(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=50
)

plt.plot(tloss)
plt.xlabel('Epoch')
plt.ylabel('Epoch_Loss')
plt.legend()
plt.show()